## Cell 1: Setup
**What this demonstrates**: Environment initialisation for Modular RAG — two Anthropic models (Haiku for reranking, Sonnet for generation), OpenAI embeddings for the vector retriever, BM25 for the sparse retriever, and Chroma as the vector store. Both retrievers are built from the same document set so swaps are a fair comparison.
**Expected output**: Setup confirmation with model names, retrieval constants, and a note on the Protocol-based interface approach.

In [ ]:
%pip install -q langchain langchain-community langchain-openai chromadb anthropic openai rank-bm25 python-dotenv

import os
import re
import json
import time
import pathlib
from dataclasses import dataclass
from typing import Protocol, runtime_checkable, TypedDict

from dotenv import load_dotenv, find_dotenv
from anthropic import Anthropic
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(find_dotenv())
_anthropic_key = os.environ.get('ANTHROPIC_API_KEY', '')
_openai_key    = os.environ.get('OPENAI_API_KEY', '')
assert _anthropic_key, 'ANTHROPIC_API_KEY not set — add it to .env'
assert _openai_key,    'OPENAI_API_KEY not set — add it to .env'

RERANK_MODEL  = 'claude-haiku-4-5-20251001'  # Fast, cheap — one call per rerank
ANSWER_MODEL  = 'claude-sonnet-4-6'           # Careful — final answer generation
EMBED_MODEL   = 'text-embedding-3-small'
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 60
K_RETRIEVE    = 6    # Chunks fetched by retriever
TOP_N_RERANK  = 3    # Chunks passed to generator after reranking

client        = Anthropic()
lc_embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

print('Setup complete')
print(f'  Rerank model    : {RERANK_MODEL}')
print(f'  Answer model    : {ANSWER_MODEL}')
print(f'  Embed model     : {EMBED_MODEL}')
print(f'  K retrieve      : {K_RETRIEVE} → top {TOP_N_RERANK} after reranking')
print()
print('Interface approach: typing.Protocol — no inheritance required.')
print('Any class with the right method signature satisfies the contract.')

## Cell 2: Data — All Sample Documents
**What this demonstrates**: Loading all four fintech sample documents into a shared chunk pool. Each chunk carries `source` metadata so the generator can cite which document an answer came from. Both the BM25 and Chroma retrievers index the same chunks — the only variable in Cell 4 vs Cell 5 is the retriever implementation.
**Expected output**: Chunk counts per document and total, with a preview of the metadata structure.

In [ ]:
_base_candidates = [
    pathlib.Path('../../shared/sample_data'),
    pathlib.Path('shared/sample_data'),
]
BASE_DIR = next((p for p in _base_candidates if p.exists()), None)
assert BASE_DIR is not None, 'Cannot find shared/sample_data'

DOC_FILES = {
    'basel_iii'   : BASE_DIR / 'basel_iii_excerpt.txt',
    'isda'        : BASE_DIR / 'isda_excerpt.txt',
    'policy'      : BASE_DIR / 'fintech_policy.txt',
    'earnings'    : BASE_DIR / 'earnings_report.txt',
}

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '.', ' '],
)

all_chunk_texts: list[str]  = []
all_chunk_meta:  list[dict] = []
all_lc_docs:     list[Document] = []

for source_name, path in DOC_FILES.items():
    raw    = path.read_text(encoding='utf-8')
    chunks = splitter.create_documents([raw])
    for idx, chunk in enumerate(chunks):
        meta = {'source': source_name, 'chunk_idx': idx}
        all_chunk_texts.append(chunk.page_content)
        all_chunk_meta.append(meta)
        all_lc_docs.append(Document(page_content=chunk.page_content, metadata=meta))
    print(f'  {source_name:12s}: {len(raw):,} chars  →  {len(chunks)} chunks')

print(f'\nTotal: {len(all_chunk_texts)} chunks across {len(DOC_FILES)} documents')

# Build Chroma index for VectorRetriever (built once, shared across pipelines)
print('\nBuilding Chroma index...', end=' ', flush=True)
vectorstore = Chroma.from_documents(
    all_lc_docs, lc_embeddings, collection_name='modular_rag_all_docs'
)
print('done')

print('\nSample chunk metadata:')
print(f'  {all_chunk_meta[0]}')
print(f'  {all_chunk_meta[-1]}')

## Cell 3: Core — Protocol Interfaces, Concrete Modules, RAGPipeline
**What this demonstrates**: The full Modular RAG architecture. Three `Protocol` classes define what a retriever, reranker, and generator must do — the method signature is the contract. Four concrete classes implement those protocols. `RAGPipeline` is a dataclass that accepts any combination of the three via dependency injection and runs `retrieve → rerank → generate`. Swapping a component is replacing one constructor argument.
**Expected output**: Protocol compliance check (isinstance) and pipeline ready messages.

In [ ]:
# ── Shared data contract ───────────────────────────────────────────────────────
# RetrievedChunk is the currency between all modules. Every module reads and
# writes this type — never its own proprietary format.

class RetrievedChunk(TypedDict):
    text    : str
    score   : float   # Normalised to 0–1 where possible
    metadata: dict    # At minimum: {'source': str}


@dataclass
class PipelineResult:
    """Per-stage telemetry + final answer for observability and A/B comparison."""
    answer              : str
    chunks_retrieved    : list   # list[RetrievedChunk]
    chunks_reranked     : list   # list[RetrievedChunk] — top_n subset
    latency_retrieve_ms : float
    latency_rerank_ms   : float
    latency_generate_ms : float
    total_latency_ms    : float
    pipeline_name       : str = ''


# ── Protocol interfaces ────────────────────────────────────────────────────────
# @runtime_checkable enables isinstance(obj, RetrieverModule) — useful for tests.
# No inheritance: any class with the right method signature satisfies the protocol.

@runtime_checkable
class RetrieverModule(Protocol):
    def retrieve(self, query: str, k: int) -> list: ...

@runtime_checkable
class RerankerModule(Protocol):
    def rerank(self, query: str, chunks: list, top_n: int) -> list: ...

@runtime_checkable
class GeneratorModule(Protocol):
    def generate(self, query: str, chunks: list) -> str: ...


# ── BM25Retriever ──────────────────────────────────────────────────────────────

class BM25Retriever:
    """Sparse keyword retriever using BM25Okapi. No embedding cost."""

    def __init__(self, chunks: list[str], metadata: list[dict] | None = None):
        from rank_bm25 import BM25Okapi
        # Tokenise by lowercased words — simple and fast for domain-specific text
        self._bm25     = BM25Okapi([c.lower().split() for c in chunks])
        self._chunks   = chunks
        self._metadata = metadata or [{} for _ in chunks]

    def retrieve(self, query: str, k: int = 5) -> list:
        scores = self._bm25.get_scores(query.lower().split())
        top_k  = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        max_s  = max(scores[i] for i in top_k) or 1.0   # Avoid division by zero
        return [
            {'text': self._chunks[i], 'score': scores[i] / max_s, 'metadata': self._metadata[i]}
            for i in top_k
        ]


# ── VectorRetriever ────────────────────────────────────────────────────────────

class VectorRetriever:
    """Dense semantic retriever backed by Chroma. Excels at paraphrase matching."""

    def __init__(self, vs: Chroma):
        self._vs = vs

    def retrieve(self, query: str, k: int = 5) -> list:
        results = self._vs.similarity_search_with_score(query, k=k)
        # Chroma returns L2 distances; convert to similarity: 1 / (1 + dist)
        return [
            {'text': doc.page_content, 'score': 1.0 / (1.0 + dist), 'metadata': doc.metadata}
            for doc, dist in results
        ]


# ── LLMReranker ───────────────────────────────────────────────────────────────

class LLMReranker:
    """Reranks chunks with a single RERANK_MODEL call that returns a ranked index list.

    One call for all chunks is cheaper than per-chunk scoring while still
    leveraging LLM reasoning about relevance.
    """

    def rerank(self, query: str, chunks: list, top_n: int = 3) -> list:
        if not chunks:
            return []
        # Truncate each chunk preview to keep the prompt size predictable
        numbered = '\n\n'.join(
            f'[{i+1}] {c["text"][:250]}' for i, c in enumerate(chunks)
        )
        prompt = (
            f'Query: {query}\n\n'
            f'Chunks:\n{numbered}\n\n'
            f'Return a JSON array of chunk numbers ranked by relevance to the query, '
            f'most relevant first. Example: [3, 1, 4]. Return only the JSON array.'
        )
        response = client.messages.create(
            model=RERANK_MODEL,
            max_tokens=100,
            messages=[{'role': 'user', 'content': prompt}],
        )
        raw   = response.content[0].text.strip()
        match = re.search(r'\[[\d,\s]+\]', raw)
        if match:
            try:
                # Convert 1-indexed positions from LLM to 0-indexed
                order   = [int(x) - 1 for x in json.loads(match.group())]
                valid   = [i for i in order if 0 <= i < len(chunks)]
                # Assign positional scores: rank 0 → 1.0, rank 1 → 0.9, …
                reranked = []
                for rank, idx in enumerate(valid[:top_n]):
                    chunk = dict(chunks[idx])
                    chunk['score'] = 1.0 - (rank * 0.1)
                    reranked.append(chunk)
                return reranked
            except (json.JSONDecodeError, ValueError):
                pass
        # Fallback: return top_n by original retrieval score
        return sorted(chunks, key=lambda c: c['score'], reverse=True)[:top_n]


# ── ClaudeGenerator ───────────────────────────────────────────────────────────

class ClaudeGenerator:
    """Answer generator using ANSWER_MODEL with retrieved + reranked context."""

    def __init__(self, model: str = ANSWER_MODEL):
        self._model = model

    def generate(self, query: str, chunks: list) -> str:
        context = '\n\n'.join(
            f'[{c["metadata"].get("source", "doc")}]\n{c["text"]}' for c in chunks
        )
        response = client.messages.create(
            model=self._model,
            max_tokens=400,
            system='Answer the query using only the provided context. Cite the source label in brackets.',
            messages=[{'role': 'user', 'content': f'Context:\n{context}\n\nQuery: {query}'}],
        )
        return response.content[0].text.strip()


# ── RAGPipeline ────────────────────────────────────────────────────────────────

@dataclass
class RAGPipeline:
    """Assembles three modules via dependency injection and runs the full pipeline.

    The pipeline holds references to protocol interfaces — it has no knowledge
    of which concrete class is injected. Swapping a module is replacing one
    constructor argument.
    """
    retriever    : object   # Must satisfy RetrieverModule
    reranker     : object   # Must satisfy RerankerModule
    generator    : object   # Must satisfy GeneratorModule
    name         : str = 'pipeline'
    k_retrieve   : int = K_RETRIEVE
    top_n_rerank : int = TOP_N_RERANK

    def run(self, query: str) -> PipelineResult:
        t0 = time.perf_counter()

        t1        = time.perf_counter()
        retrieved = self.retriever.retrieve(query, k=self.k_retrieve)
        t_ret     = (time.perf_counter() - t1) * 1000

        t2        = time.perf_counter()
        reranked  = self.reranker.rerank(query, retrieved, top_n=self.top_n_rerank)
        t_rer     = (time.perf_counter() - t2) * 1000

        t3        = time.perf_counter()
        answer    = self.generator.generate(query, reranked)
        t_gen     = (time.perf_counter() - t3) * 1000

        return PipelineResult(
            answer               = answer,
            chunks_retrieved     = retrieved,
            chunks_reranked      = reranked,
            latency_retrieve_ms  = t_ret,
            latency_rerank_ms    = t_rer,
            latency_generate_ms  = t_gen,
            total_latency_ms     = (time.perf_counter() - t0) * 1000,
            pipeline_name        = self.name,
        )


# ── Instantiate concrete modules and verify protocol compliance ────────────────

bm25_retriever   = BM25Retriever(all_chunk_texts, all_chunk_meta)
vector_retriever = VectorRetriever(vectorstore)
llm_reranker     = LLMReranker()
claude_generator = ClaudeGenerator()

print('Protocol compliance (isinstance checks):')
print(f'  BM25Retriever   satisfies RetrieverModule : {isinstance(bm25_retriever,   RetrieverModule)}')
print(f'  VectorRetriever satisfies RetrieverModule : {isinstance(vector_retriever, RetrieverModule)}')
print(f'  LLMReranker     satisfies RerankerModule  : {isinstance(llm_reranker,     RerankerModule)}')
print(f'  ClaudeGenerator satisfies GeneratorModule : {isinstance(claude_generator, GeneratorModule)}')
print()
print('Pipelines ready to assemble via RAGPipeline(retriever, reranker, generator).')
print('Swap any component by passing a different concrete class — nothing else changes.')

## Cell 4: Run — BM25 Pipeline
**What this demonstrates**: The BM25 retriever pipeline end-to-end. BM25 matches on exact terms — strong for precise regulatory vocabulary ("capital adequacy", "CET1", "leverage ratio") that appears verbatim in the documents.
**Expected output**: Retrieved chunks with BM25 scores, post-rerank top-3, final answer, and per-stage latency breakdown.

In [ ]:
QUERY = 'What are the Basel III capital adequacy requirements?'

# Assemble the BM25 pipeline — three lines: one module each
bm25_pipeline = RAGPipeline(
    retriever = bm25_retriever,
    reranker  = llm_reranker,
    generator = claude_generator,
    name      = 'BM25 + LLMReranker + Sonnet',
)

bm25_result = bm25_pipeline.run(QUERY)

print(f'Pipeline  : {bm25_result.pipeline_name}')
print(f'Query     : {QUERY}')
print()

print(f'Retrieved {len(bm25_result.chunks_retrieved)} chunks (BM25 scores):')
print('-' * 65)
for i, c in enumerate(bm25_result.chunks_retrieved):
    src = c['metadata'].get('source', 'unknown')
    print(f'  [{i+1}] score={c["score"]:.3f}  source={src}')
    print(f'      {c["text"][:100].replace(chr(10), " ")}...')

print()
print(f'After rerank → top {TOP_N_RERANK}:')
print('-' * 65)
for i, c in enumerate(bm25_result.chunks_reranked):
    src = c['metadata'].get('source', 'unknown')
    print(f'  Rank {i+1}  rerank_score={c["score"]:.2f}  source={src}')

print()
print('Final answer:')
print('=' * 65)
print(bm25_result.answer)
print()
print('Latency breakdown:')
print(f'  Retrieve (BM25)       : {bm25_result.latency_retrieve_ms:.0f} ms')
print(f'  Rerank  (LLM)         : {bm25_result.latency_rerank_ms:.0f} ms')
print(f'  Generate (Sonnet)     : {bm25_result.latency_generate_ms:.0f} ms')
print(f'  Total                 : {bm25_result.total_latency_ms:.0f} ms')

## Cell 5: Inspect — Swap to Vector Retriever, Compare
**What this demonstrates**: The core value of Modular RAG — one line changes the retriever, everything else stays identical. VectorRetriever matches semantically, surfacing paraphrased or conceptually related chunks that BM25 misses. The comparison reveals which chunks each approach surfaces and whether the final answer differs.
**Expected output**: Vector pipeline results, then a side-by-side comparison: chunk overlap, source distribution, answer delta, and latency per stage.

In [ ]:
# ── Swap the retriever — one constructor argument changes ─────────────────────
vector_pipeline = RAGPipeline(
    retriever = vector_retriever,   # ← only this line changes
    reranker  = llm_reranker,
    generator = claude_generator,
    name      = 'Vector + LLMReranker + Sonnet',
)

vector_result = vector_pipeline.run(QUERY)

print(f'Pipeline  : {vector_result.pipeline_name}')
print(f'Query     : {QUERY}')
print()
print(f'Retrieved {len(vector_result.chunks_retrieved)} chunks (vector similarity scores):')
print('-' * 65)
for i, c in enumerate(vector_result.chunks_retrieved):
    src = c['metadata'].get('source', 'unknown')
    print(f'  [{i+1}] score={c["score"]:.3f}  source={src}')
    print(f'      {c["text"][:100].replace(chr(10), " ")}...')

print()
print(f'After rerank → top {TOP_N_RERANK}:')
for i, c in enumerate(vector_result.chunks_reranked):
    src = c['metadata'].get('source', 'unknown')
    print(f'  Rank {i+1}  rerank_score={c["score"]:.2f}  source={src}')

print()
print('Final answer:')
print('=' * 65)
print(vector_result.answer)

# ── Side-by-side comparison ────────────────────────────────────────────────────
print()
print('COMPARISON: BM25 vs VECTOR')
print('=' * 65)

bm25_texts   = {c['text'][:80] for c in bm25_result.chunks_retrieved}
vector_texts = {c['text'][:80] for c in vector_result.chunks_retrieved}
shared       = bm25_texts & vector_texts
bm25_only    = bm25_texts - vector_texts
vector_only  = vector_texts - bm25_texts

print(f'Retrieved chunks — overlap: {len(shared)}/{K_RETRIEVE}')
print(f'  BM25-only  : {len(bm25_only)} chunks  (exact keyword matches not in semantic space)')
print(f'  Vector-only: {len(vector_only)} chunks  (semantic matches without exact terms)')

# Source distribution per pipeline
from collections import Counter
bm25_sources   = Counter(c['metadata'].get('source', '?') for c in bm25_result.chunks_retrieved)
vector_sources = Counter(c['metadata'].get('source', '?') for c in vector_result.chunks_retrieved)

print()
print('Source distribution (retrieved):')
all_sources = sorted(set(bm25_sources) | set(vector_sources))
print(f'  {"Source":14s}  BM25  Vector')
print(f'  {"-"*14}  ----  ------')
for s in all_sources:
    print(f'  {s:14s}  {bm25_sources.get(s, 0):4d}  {vector_sources.get(s, 0):6d}')

# Latency comparison
print()
print('Latency (ms):')
print(f'  {"Stage":22s}  BM25  Vector')
print(f'  {"-"*22}  ----  ------')
print(f'  {"Retrieve":22s}  {bm25_result.latency_retrieve_ms:4.0f}  {vector_result.latency_retrieve_ms:6.0f}')
print(f'  {"Rerank":22s}  {bm25_result.latency_rerank_ms:4.0f}  {vector_result.latency_rerank_ms:6.0f}')
print(f'  {"Generate":22s}  {bm25_result.latency_generate_ms:4.0f}  {vector_result.latency_generate_ms:6.0f}')
print(f'  {"Total":22s}  {bm25_result.total_latency_ms:4.0f}  {vector_result.total_latency_ms:6.0f}')
print()
print('Key insight: same reranker, same generator — any answer quality difference')
print('is attributable to retrieval alone. This is A/B testing at the module level.')

## Cell 6: Fintech — Modular Architecture for Multi-Domain Knowledge Base
**What this demonstrates**: A compliance knowledge base serving three query domains — Basel III regulation, ISDA contract terms, and earnings data — each routed to a pipeline configuration optimised for that domain's retrieval characteristics. Regulatory queries use the vector retriever (semantic matching across paraphrased rule language); contract queries use BM25 (legal terms appear verbatim); earnings queries use vector retrieval with a fast Haiku generator (speed matters, summaries acceptable). All three pipelines share the same `RAGPipeline` interface — only the injected modules differ.
**Expected output**: Three domain queries, per-domain pipeline selection, answers with source citations, and an aggregate configuration table.

In [ ]:
# ── Domain-specific pipeline configurations ────────────────────────────────────
# Same RAGPipeline class, different module combinations.
# The domain router selects the pipeline; no pipeline knows about routing.

haiku_generator = ClaudeGenerator(model=RERANK_MODEL)   # Haiku — fast summaries

domain_pipelines = {
    'regulatory': RAGPipeline(
        retriever = vector_retriever,   # Semantic — regulatory language is paraphrased
        reranker  = llm_reranker,
        generator = claude_generator,   # Sonnet — precision matters for compliance
        name      = 'regulatory (Vector + LLMReranker + Sonnet)',
    ),
    'contract': RAGPipeline(
        retriever = bm25_retriever,     # BM25 — legal terms appear verbatim in docs
        reranker  = llm_reranker,
        generator = claude_generator,   # Sonnet — contract interpretation needs care
        name      = 'contract (BM25 + LLMReranker + Sonnet)',
    ),
    'earnings': RAGPipeline(
        retriever = vector_retriever,   # Semantic — analyst questions vary in phrasing
        reranker  = llm_reranker,
        generator = haiku_generator,    # Haiku — fast summaries, latency-sensitive
        name      = 'earnings (Vector + LLMReranker + Haiku)',
    ),
}

# ── Domain queries ─────────────────────────────────────────────────────────────

domain_queries = [
    ('regulatory', 'What is the minimum CET1 capital ratio under Basel III?'),
    ('contract',   'What are the close-out netting provisions in the ISDA agreement?'),
    ('earnings',   'What were the key financial highlights for Q3?'),
]

domain_results = []

for domain, query in domain_queries:
    pipeline = domain_pipelines[domain]
    print(f'Domain   : {domain.upper()}')
    print(f'Pipeline : {pipeline.name}')
    print(f'Query    : {query}')
    print()

    result = pipeline.run(query)
    domain_results.append((domain, query, result))

    sources = [c['metadata'].get('source', '?') for c in result.chunks_reranked]
    print(f'Sources cited: {", ".join(sources)}')
    print('Answer:')
    print('-' * 65)
    print(result.answer)
    print()
    print(f'Latency: retrieve={result.latency_retrieve_ms:.0f}ms  '
          f'rerank={result.latency_rerank_ms:.0f}ms  '
          f'generate={result.latency_generate_ms:.0f}ms  '
          f'total={result.total_latency_ms:.0f}ms')
    print()
    print('=' * 65)
    print()

# ── Configuration summary table ────────────────────────────────────────────────
print('PIPELINE CONFIGURATION SUMMARY')
print('=' * 65)
print(f'{"Domain":12s}  {"Retriever":14s}  {"Generator":8s}  {"Total ms":8s}  Sources')
print(f'{"-"*12}  {"-"*14}  {"-"*8}  {"-"*8}  -------')
for domain, query, result in domain_results:
    pipeline = domain_pipelines[domain]
    ret_type = 'BM25' if isinstance(pipeline.retriever, BM25Retriever) else 'Vector'
    gen_type = 'Haiku' if pipeline.generator._model == RERANK_MODEL else 'Sonnet'
    srcs     = ','.join(sorted({c['metadata'].get('source', '?')
                                 for c in result.chunks_reranked}))
    print(f'{domain:12s}  {ret_type:14s}  {gen_type:8s}  {result.total_latency_ms:8.0f}  {srcs}')

print()
print('All three pipelines use the same RAGPipeline class.')
print('Component selection is a constructor argument — no conditional logic in the pipeline.')
print('To add a new domain: define a new RAGPipeline instance and add it to domain_pipelines.')